Environment SetUp

In [3]:
import math
import pathlib
import sys
import torch
import torch.nn as nn 
import torch.nn.functional as F

#find the repo root

import sys
from pathlib import Path 
REPO_ROOT=Path.cwd().resolve().parents[0]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0,str(REPO_ROOT))
print(REPO_ROOT)
import sys
sys.path.append("../..")

/home/ghada/Documents/custom-llm-architecture/llm_Embedding/assets


In [4]:
import importlib
import model.config
import model.attention
importlib.reload(model.config)
importlib.reload(model.attention)

from model.config import ModelConfig
from model.config import ModelConfig
TOKENS=["the","cat","sat","on","mat"]
cfg=ModelConfig(
    vocab_size=32,
    d_model=8,
    n_heads=2,
    n_layers=2,
    max_seq_len=16,
    dropout=0.1,
)
B=1
T=len(TOKENS)
x=torch.randn(B,T,cfg.d_model)

print("d_model    :", cfg.d_model)
print("n_heads    :", cfg.n_heads, " -> d_k =", cfg.d_k)
print("n_layers   :", cfg.n_layers)
print("d_ff       :", cfg.d_ff, " <- __post_init__ filled this in: 4 * d_model")
print("vocab_size :", cfg.vocab_size)
print("max_seq_len:", cfg.max_seq_len)



d_model    : 8
n_heads    : 2  -> d_k = 4
n_layers   : 2
d_ff       : 3072  <- __post_init__ filled this in: 4 * d_model
vocab_size : 32
max_seq_len: 16


In [6]:
from model.attention import MultiHeadAttention
from model.feedforward import FeedForward
attn=MultiHeadAttention(cfg)
ffn=FeedForward(cfg)
attn_out,updated_cache=attn(x)
y=ffn(attn_out)
print("x:",tuple(x.shape))
print("attn:",tuple(attn_out.shape))
print("ffn:",tuple(y.shape))
print("same shape in and out?",tuple(y.shape)==tuple(x.shape))

x: (1, 5, 8)
attn: (1, 5, 8)
ffn: (1, 5, 8)
same shape in and out? True


In [10]:
ln=nn.LayerNorm(cfg.d_model)
wild_input=torch.tensor([
    [
        [50.0, 52.0, 48.0, 51.0, 49.0, 53.0, 47.0, 50.0]
    ]
])

normed_input = ln(wild_input)

print(wild_input.shape)
print(normed_input.shape)

print(wild_input)
print(normed_input)
print("wild_input: ")
print("  mean:", round(wild_input.mean().item(), 3),
      "  std:", round(wild_input.std(unbiased=False).item(), 3))
print()

print("normed_input: ")
print("  mean:", round(normed_input.mean().item(), 3),
      "  std:", round(normed_input.std(unbiased=False).item(), 3))
print()

torch.Size([1, 1, 8])
torch.Size([1, 1, 8])
tensor([[[50., 52., 48., 51., 49., 53., 47., 50.]]])
tensor([[[ 0.0000,  1.0690, -1.0690,  0.5345, -0.5345,  1.6036, -1.6036,
           0.0000]]], grad_fn=<NativeLayerNormBackward0>)
wild_input: 
  mean: 50.0   std: 1.871

normed_input: 
  mean: 0.0   std: 1.0



In [ ]:
from model.attention import MultiHeadAttention
from model.feedforward import FeedForward
from model.block import TransformerBlock

attn = MultiHeadAttention(cfg)
ffn = FeedForward(cfg)

attn_out, updated_cache = attn(x)
y = ffn(attn_out)

blocks = nn.ModuleList(
    TransformerBlock(cfg)
    for ix in range(cfg.n_layers)
)
y.shape # (B, T, d_model)